In [6]:
"""
Automated Product Listing Generator
Author: Jay
Description: Generate e-commerce product listings from images using ChatGPT Vision
"""

%pip install openai python-dotenv pillow requests --quiet

Note: you may need to restart the kernel to use updated packages.


In [21]:
import os
import json
import time
import base64
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv()

# Retrieve the API key securely
api_key = os.getenv("OPENAI_API_KEY")

# Sanity check WITHOUT ever printing the actual key
if api_key:
    print(f"API key loaded successfully! (starts with: {api_key[:7]}..., length: {len(api_key)} characters)")
else:
    print("⚠ API key not found. Check your .env file exists and is in the same folder as this notebook.")

# Initialize the OpenAI client
client = OpenAI(api_key=api_key)
print("OpenAI client initialized!")

API key loaded successfully! (starts with: sk-svca..., length: 167 characters)
OpenAI client initialized!


## Step 2: Preparing the Dataset

Rather than pulling the full HuggingFace fashion dataset, I'm starting with a small set of 3 products spanning different categories: Electronics, Fashion & Accessories, and Home & Kitchen.

**Why start small:** Each API call to GPT-4 Vision costs money and takes time. Since I expect few iterations before perfect output, testing on a full batch before validating the pipeline and prompt quality would risk wasting API credits. So I start by proving the pipeline works on a small sample, refine based on output and then scale.

Testing across three distinct categories also lets me check whether a single prompt template generalizes well, rather than only working for one narrow product type.

In [8]:
import pandas as pd
from pathlib import Path

# Build a small product dataset, starting with 3 products.
IMAGES_DIR = Path("product_images")

products_data = [
    {
        "id": 1,
        "name": "Wireless Bluetooth Headphones",
        "price": 79.99,
        "category": "Electronics",
        "additional_info": "Noise cancelling, 30-hour battery life",
        "image_path": IMAGES_DIR / "headphones.jpg"
    },
    {
        "id": 2,
        "name": "Leather Travel Backpack",
        "price": 129.99,
        "category": "Fashion & Accessories",
        "additional_info": "Genuine leather, fits 15-inch laptop",
        "image_path": IMAGES_DIR / "backpack.jpg"
    },
    {
        "id": 3,
        "name": "Stainless Steel Water Bottle",
        "price": 24.99,
        "category": "Home & Kitchen",
        "additional_info": "Double-wall insulated, keeps drinks cold 24 hours",
        "image_path": IMAGES_DIR / "water_bottle.jpg"
    }
]

products_df = pd.DataFrame(products_data)

print(f"✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")
print(f"\n{products_df[['id', 'name', 'price', 'category']]}")

# Verify all image files actually exist before we try to use them
print("\nImage file check:")
for _, row in products_df.iterrows():
    exists = row['image_path'].exists()
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status} {row['name']}: {row['image_path']}")

✓ Dataset prepared!
  Total products: 3

   id                           name   price               category
0   1  Wireless Bluetooth Headphones   79.99            Electronics
1   2        Leather Travel Backpack  129.99  Fashion & Accessories
2   3   Stainless Steel Water Bottle   24.99         Home & Kitchen

Image file check:
  ✓ Wireless Bluetooth Headphones: product_images\headphones.jpg
  ✓ Leather Travel Backpack: product_images\backpack.jpg
  ✓ Stainless Steel Water Bottle: product_images\water_bottle.jpg


In [11]:
import base64

def encode_image_to_base64(image_path):
    """Encode an image file to base64 string."""
    with open(image_path, "rb") as img_file:
        encoded = base64.b64encode(img_file.read()).decode("utf-8")
    return encoded

# Test on the first product
sample_path = products_df.iloc[0]["image_path"]
encoded_image = encode_image_to_base64(sample_path)

print(f"Product: {products_df.iloc[0]['name']}")
print(f"Encoded image length: {len(encoded_image)} characters")
print(f"Encoded prefix: {encoded_image[:40]}...")

Product: Wireless Bluetooth Headphones
Encoded image length: 35432 characters
Encoded prefix: /9j/4AAQSkZJRgABAQEBLAEsAAD/7QDGUGhvdG9z...


In [12]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}

Please create a professional product listing that includes:

1. Product Title (catchy, SEO-friendly, 60 characters max)
2. Product Description (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. Key Features (5-7 bullet points)
4. SEO Keywords (10-15 relevant keywords)

Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt

# Test prompt creation on our first product
test_row = products_df.iloc[0]
test_prompt = create_product_listing_prompt(
    product_name=test_row["name"],
    price=test_row["price"],
    category=test_row["category"],
    additional_info=test_row["additional_info"]
)

print("="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(test_prompt)

PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery life

Please create a professional product listing that includes:

1. Product Title (catchy, SEO-friendly, 60 characters max)
2. Product Description (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. Key Features (5-7 bullet points)
4. SEO Keywords (10-15 relevant keywords)

Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features.


In [22]:
from pydantic import BaseModel
from typing import List

class ProductListing(BaseModel):
    title: str
    description: str
    features: List[str]
    keywords: List[str]

def generate_product_listing(product_row):
    """
    Call GPT-4 Vision to generate a product listing for a single product.
    Returns a ProductListing object, or None if the call fails.
    """
    try:
        # Encode the image
        encoded_image = encode_image_to_base64(product_row["image_path"])
        
        # Build the prompt
        prompt = create_product_listing_prompt(
            product_name=product_row["name"],
            price=product_row["price"],
            category=product_row["category"],
            additional_info=product_row.get("additional_info")
        )
        
        # Call the API with the image + prompt, forcing structured output
        completion = client.chat.completions.parse(
            model="gpt-4o-2024-08-06",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{encoded_image}"
                            }
                        }
                    ]
                }
            ],
            response_format=ProductListing,
        )
        
        return completion.choices[0].message.parsed
    
    except Exception as e:
        print(f"⚠ Error generating listing for {product_row['name']}: {e}")
        return None

# TEST: generate a listing for just ONE product first
print("Generating listing for first product (this may take 10-20 seconds)...\n")
test_listing = generate_product_listing(products_df.iloc[0])

if test_listing:
    print("✓ SUCCESS!\n")
    print(f"TITLE: {test_listing.title}\n")
    print(f"DESCRIPTION:\n{test_listing.description}\n")
    print(f"FEATURES:")
    for feature in test_listing.features:
        print(f"  • {feature}")
    print(f"\nKEYWORDS: {', '.join(test_listing.keywords)}")
else:
    print("✗ Generation failed — check error above")

Generating listing for first product (this may take 10-20 seconds)...

✓ SUCCESS!

TITLE: Premium Noise-Cancelling Bluetooth Headphones

DESCRIPTION:
Elevate your listening experience with our Wireless Bluetooth Headphones. Designed for music lovers who demand excellence, these headphones offer advanced noise-cancelling technology that immerses you in pure, uninterrupted sound. Crafted with sleek, matte black materials, they boast a modern design that complements any style. The lightweight, ergonomic construction ensures comfort during prolonged use, making them perfect for long commutes or extended listening sessions.

Enjoy an impressive 30-hour battery life, allowing you to keep the music playing all day and night without frequent recharging. Whether you're taking calls or jamming to your favorite tunes, the intuitive controls on the earcups provide seamless operation. Embrace the freedom of wireless connectivity, and relish in audio that moves with you.

FEATURES:
  • Advanced nois

In [7]:
import json
import time

def process_all_products(products_df):
    """
    Generate listings for all products in the dataframe.
    Returns a list of results (dict per product), handles errors gracefully.
    """
    results = []
    
    for idx, row in products_df.iterrows():
        print(f"[{idx + 1}/{len(products_df)}] Processing: {row['name']}...")
        
        listing = generate_product_listing(row)
        
        if listing:
            result = {
                "id": int(row["id"]),
                "name": row["name"],
                "price": float(row["price"]),
                "category": row["category"],
                "title": listing.title,
                "description": listing.description,
                "features": listing.features,
                "keywords": listing.keywords,
                "status": "success"
            }
            print(f"  ✓ Success")
        else:
            result = {
                "id": int(row["id"]),
                "name": row["name"],
                "price": float(row["price"]),
                "category": row["category"],
                "status": "failed"
            }
            print(f"  ✗ Failed")
        
        results.append(result)
        
        # Small delay between calls to be respectful of rate limits
        time.sleep(1)
    
    return results

# Process all products
print("="*50)
print("BATCH PROCESSING PRODUCTS")
print("="*50)
all_results = process_all_products(products_df)

# Save results to JSON immediately, so nothing is lost
output_file = "generated_listings.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)

successful = sum(1 for r in all_results if r["status"] == "success")
print(f"\n{'='*50}")
print(f"BATCH COMPLETE: {successful}/{len(all_results)} successful")
print(f"Results saved to {output_file}")

BATCH PROCESSING PRODUCTS
[1/3] Processing: Wireless Bluetooth Headphones...
  ✓ Success
[2/3] Processing: Leather Travel Backpack...
  ✓ Success
[3/3] Processing: Stainless Steel Water Bottle...
  ✓ Success

BATCH COMPLETE: 3/3 successful
Results saved to generated_listings.json


In [14]:
%pip install datasets --quiet

Note: you may need to restart the kernel to use updated packages.


In [15]:
from datasets import load_dataset

print("Loading HuggingFace fashion product dataset...")
try:
    hf_dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:20]")  # Just first 20, not full 44K
    print(f"✓ Loaded {len(hf_dataset)} products")
    print(f"\nAvailable columns: {hf_dataset.column_names}")
    print(f"\nFirst item structure:")
    print(hf_dataset[0])
except Exception as e:
    print(f"⚠ Could not load dataset: {e}")

Loading HuggingFace fashion product dataset...
✓ Loaded 20 products

Available columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

First item structure:
{'id': 15970, 'gender': 'Men', 'masterCategory': 'Apparel', 'subCategory': 'Topwear', 'articleType': 'Shirts', 'baseColour': 'Navy Blue', 'season': 'Fall', 'year': 2011.0, 'usage': 'Casual', 'productDisplayName': 'Turtle Check Men Navy Blue Shirt', 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=60x80 at 0x1D54038C090>}


In [16]:
# Look at the first few products to pick good candidates
for i in range(len(hf_dataset)):
    item = hf_dataset[i]
    print(f"{i}: {item['productDisplayName']} | {item['masterCategory']} > {item['subCategory']} | {item['baseColour']} | Image size: {item['image'].size}")

0: Turtle Check Men Navy Blue Shirt | Apparel > Topwear | Navy Blue | Image size: (60, 80)
1: Peter England Men Party Blue Jeans | Apparel > Bottomwear | Blue | Image size: (60, 80)
2: Titan Women Silver Watch | Accessories > Watches | Silver | Image size: (60, 80)
3: Manchester United Men Solid Black Track Pants | Apparel > Bottomwear | Black | Image size: (60, 80)
4: Puma Men Grey T-shirt | Apparel > Topwear | Grey | Image size: (60, 80)
5: Inkfruit Mens Chain Reaction T-shirt | Apparel > Topwear | Grey | Image size: (60, 80)
6: Fabindia Men Striped Green Shirt | Apparel > Topwear | Green | Image size: (60, 80)
7: Jealous 21 Women Purple Shirt | Apparel > Topwear | Purple | Image size: (60, 80)
8: Puma Men Pack of 3 Socks | Accessories > Socks | Navy Blue | Image size: (60, 80)
9: Skagen Men Black Watch | Accessories > Watches | Black | Image size: (60, 80)
10: Puma Men Future Cat Remix SF Black Casual Shoes | Footwear > Shoes | Black | Image size: (60, 80)
11: Fossil Women Black Hua

In [17]:
import pandas as pd
def encode_pil_image_to_base64(pil_image):
    """Encode a PIL Image object (in memory) to base64 string — 
    variant of encode_image_to_base64 for HuggingFace dataset images."""
    import io
    buffer = io.BytesIO()
    # Convert to RGB first in case of any transparency/mode issues, then save as JPEG into memory
    pil_image.convert("RGB").save(buffer, format="JPEG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return encoded

# Select 4 varied products from the HuggingFace dataset
selected_indices = [2, 10, 19, 6]

hf_products = []
for i, idx in enumerate(selected_indices):
    item = hf_dataset[idx]
    hf_products.append({
        "id": 100 + i,  # offset IDs so they don't collide with our first 3 products
        "name": item["productDisplayName"],
        "price": round(19.99 + (idx * 7.5), 2),  # placeholder price since dataset has none
        "category": f"{item['masterCategory']} > {item['subCategory']}",
        "additional_info": f"Color: {item['baseColour']}, Usage: {item['usage']}",
        "image": item["image"]  # PIL object, not a path this time
    })

hf_products_df = pd.DataFrame(hf_products)
print(hf_products_df[["id", "name", "price", "category"]])

    id                                             name   price  \
0  100                         Titan Women Silver Watch   34.99   
1  101  Puma Men Future Cat Remix SF Black Casual Shoes   94.99   
2  102                       Baggit Women Brown Handbag  162.49   
3  103                 Fabindia Men Striped Green Shirt   64.99   

                category  
0  Accessories > Watches  
1       Footwear > Shoes  
2     Accessories > Bags  
3      Apparel > Topwear  


In [23]:
import time
def generate_product_listing_pil(product_row):
    """
    Variant of generate_product_listing for HuggingFace dataset products,
    where the image is a PIL object already in memory rather than a file path.
    """
    try:
        # Encode the PIL image (no file path needed)
        encoded_image = encode_pil_image_to_base64(product_row["image"])
        
        prompt = create_product_listing_prompt(
            product_name=product_row["name"],
            price=product_row["price"],
            category=product_row["category"],
            additional_info=product_row.get("additional_info")
        )
        
        completion = client.chat.completions.parse(
            model="gpt-4o-2024-08-06",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{encoded_image}"
                            }
                        }
                    ]
                }
            ],
            response_format=ProductListing,
        )
        
        return completion.choices[0].message.parsed
    
    except Exception as e:
        print(f"⚠ Error generating listing for {product_row['name']}: {e}")
        return None


def process_hf_products(products_df):
    """Batch process HuggingFace-sourced products."""
    results = []
    
    for idx, row in products_df.iterrows():
        print(f"[{idx + 1}/{len(products_df)}] Processing: {row['name']}...")
        
        listing = generate_product_listing_pil(row)
        
        if listing:
            result = {
                "id": int(row["id"]),
                "name": row["name"],
                "price": float(row["price"]),
                "category": row["category"],
                "title": listing.title,
                "description": listing.description,
                "features": listing.features,
                "keywords": listing.keywords,
                "status": "success",
                "source": "huggingface"
            }
            print(f"  ✓ Success")
        else:
            result = {
                "id": int(row["id"]),
                "name": row["name"],
                "price": float(row["price"]),
                "category": row["category"],
                "status": "failed",
                "source": "huggingface"
            }
            print(f"  ✗ Failed")
        
        results.append(result)
        time.sleep(1)
    
    return results

print("="*50)
print("BATCH PROCESSING HUGGINGFACE PRODUCTS")
print("="*50)
hf_results = process_hf_products(hf_products_df)

# Append to existing results file rather than overwrite
with open("generated_listings.json", "r", encoding="utf-8") as f:
    existing_results = json.load(f)

all_combined = existing_results + hf_results

with open("generated_listings.json", "w", encoding="utf-8") as f:
    json.dump(all_combined, f, indent=2)

successful_hf = sum(1 for r in hf_results if r["status"] == "success")
print(f"\nHuggingFace batch complete: {successful_hf}/{len(hf_results)} successful")
print(f"Total listings in file now: {len(all_combined)}")

BATCH PROCESSING HUGGINGFACE PRODUCTS
[1/4] Processing: Titan Women Silver Watch...
  ✓ Success
[2/4] Processing: Puma Men Future Cat Remix SF Black Casual Shoes...
  ✓ Success
[3/4] Processing: Baggit Women Brown Handbag...
  ✓ Success
[4/4] Processing: Fabindia Men Striped Green Shirt...
  ✓ Success

HuggingFace batch complete: 4/4 successful
Total listings in file now: 7


In [24]:
# ERROR HANDLING DEMONSTRATION
# Deliberately testing with a product that has a broken/nonexistent image path,
# to prove the pipeline handles failures gracefully instead of crashing.

print("="*50)
print("ERROR HANDLING DEMONSTRATION")
print("="*50)

# Create a fake product with a path that doesn't exist
broken_product = {
    "id": 999,
    "name": "Test Product (Broken Path)",
    "price": 9.99,
    "category": "Test",
    "additional_info": None,
    "image_path": Path("product_images/does_not_exist.jpg")  # deliberately invalid
}

print(f"Attempting to generate listing for: {broken_product['name']}")
print(f"Image path (intentionally invalid): {broken_product['image_path']}\n")

result = generate_product_listing(broken_product)

if result is None:
    print("✓ Error was caught gracefully — no crash occurred.")
    print("  The function returned None instead of raising an unhandled exception,")
    print("  allowing a batch process to continue past this failure.")
else:
    print("Unexpected: generation succeeded despite broken path")

ERROR HANDLING DEMONSTRATION
Attempting to generate listing for: Test Product (Broken Path)
Image path (intentionally invalid): product_images\does_not_exist.jpg

⚠ Error generating listing for Test Product (Broken Path): [Errno 2] No such file or directory: 'product_images\\does_not_exist.jpg'
✓ Error was caught gracefully — no crash occurred.
  The function returned None instead of raising an unhandled exception,
  allowing a batch process to continue past this failure.


## Report: Automated Product Listing Generator

**How the API integration works:** The pipeline loads a product image (either from a local file or a HuggingFace dataset's in-memory PIL object), encodes it to base64 and sends it with a structured prompt to GPT-4o via `client.chat.completions.parse()`. Rather than asking the model to format its own JSON (the lab template's original approach), I used Pydantic's `response_format` parameter to define the output shape (`title`, `description`, `features`, `keywords`) as a Python class. This eliminates the JSON-parsing failure mode risk.

**Challenges faced:** The main challenge wasn't the API itself but kernel state management. After installing new packages mid-session, several kernel restarts caused `NameError`s for previously-imported modules (`pandas`, `time`, `json`) in later cells. Since re-running the entire notebook would have triggered (and re-charged) the already-successful batch API calls, I had to selectively re-run only the setup cells, which required careful tracking of what state had actually been lost. This reinforced the value of testing small before scaling, protecting the successful, already-paid-for results while debugging around them.

**Quality of generated listings:** Across 7 products (3 hand-picked images, 4 from a HuggingFace dataset with 60×80px thumbnails), all generations succeeded and produced relevant listings. I verified one output against its source image (headphones described as "sleek and stylish black design") and confirmed the model's visual description was accurate, not hallucinated. The tiny HuggingFace thumbnails still produced coherent listings.

**Potential improvements:** (1) Real pricing data instead of placeholder formulas for the HuggingFace batch.(2)Side-by-side accuracy comparison between full-resolution and thumbnail images to quantify whether image quality  affects description accuracy. (3)Cost tracking per request to project expenses at the 10-20 product scale the lab originally recommended.